# String-Cleaning Helpers (`clean_bary`)

Defines `clean_bary` and helpers as **module-level functions only** -- no DataFrame
mutation here.  `clean_bary` is called inside `_compute_one` (Cell 11), which makes
the pipeline robust to cells being re-run out of order.


### 1 · Build Page List (36 pages)

In [1]:
base = "http://faculty.evansville.edu/ck6/encyclopedia/"

pages = [base + "ETC.html"] + [base + f"ETCPart{i}.html" for i in range(2, 37)]

print(f"Total pages: {len(pages)}")
pages[:3]

Total pages: 36


['http://faculty.evansville.edu/ck6/encyclopedia/ETC.html',
 'http://faculty.evansville.edu/ck6/encyclopedia/ETCPart2.html',
 'http://faculty.evansville.edu/ck6/encyclopedia/ETCPart3.html']

### 2 · HTTP Fetcher

In [2]:
import requests
import urllib3
import time

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def get_page(url, retries=3, delay=1.5):
    """Fetch a page with retry logic and polite rate-limiting."""
    for attempt in range(retries):
        try:
            r = requests.get(url, verify=False, timeout=20)
            r.raise_for_status()
            time.sleep(delay)          # be polite to Evansville's server
            return r.text
        except Exception as e:
            if attempt == retries - 1:
                print(f"  FAILED {url}: {e}")
                return None
            time.sleep(delay * 2)

### 3 · HTML Parser — Extract Center Headers

In [3]:
from bs4 import BeautifulSoup, NavigableString, Tag
import re

def parse_centers(html):
    """Return list of dicts with index, name, and the h3 BS4 node."""
    soup = BeautifulSoup(html, "html.parser")
    centers = []
    for h3 in soup.find_all("h3"):
        text = h3.get_text(" ", strip=True)
        match = re.search(r"X\((\d+)\)\s*=\s*(.*)", text)
        if not match:
            continue
        idx  = int(match.group(1))
        name = match.group(2).strip()
        centers.append({"index": idx, "name": name, "node": h3})
    return centers


### 4 · Extract Barycentric Strings & Function Definitions

**v11 upgrade:** `extract_bary_and_funcs` now returns **all** Barycentrics lines found in a center's HTML block as a `list[str]`, not just the first one.

The AST Complexity Scorer (Section 11) then picks the algebraically simplest representation — favouring pure polynomial (a, b, c, SA, SB, ...) coordinates over trigonometric ones, which are ×10 more expensive in GCD computations.

In [4]:
# ── Helpers: line-by-line extraction of ETC sibling text ─────────────────────

def _node_to_lines(center_node):
    """
    Walk siblings of center_node up to the next h3/h2.
    <br> tags become line separators; <sup>/<sub> content is kept inline.
    Returns list of non-empty stripped lines.
    """
    lines = []
    buf   = []

    def flush():
        line = " ".join(buf).replace("\xa0", " ")
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)
        buf.clear()

    def visit(node):
        if isinstance(node, NavigableString):
            txt = str(node).replace("\xa0", " ").replace("\n", " ")
            if txt.strip():
                buf.append(txt.strip())
        elif isinstance(node, Tag):
            if node.name == "br":
                flush()
            elif node.name not in ("p", "hr", "h3", "h2"):
                # Recurse into inline elements (<b>, <i>, <a>, <sup>, ...)
                # but NOT into block elements that start the prose section.
                for child in node.children:
                    visit(child)

    # Walk ONLY the coordinate header block.
    # ETC HTML: Trilinears/Barycentrics/Tripolars lines come first as
    # direct text+<br> siblings of <h3>.  Prose notes follow in <p>/<hr>.
    # Stopping at <p>/<hr> prevents prose contamination from notes like
    # "Barycentrics for X(N) are ..." or "semi-major axis of ... : :".
    node = center_node.next_sibling
    while node:
        if isinstance(node, Tag) and node.name in ("h3", "h2", "hr", "p"):
            break
        visit(node)
        node = node.next_sibling

    flush()
    return lines


# Stop-words that end a Barycentrics expression
_BARY_STOP = re.compile(
    r"\s+(?:where|for\b|which\b|See\s+also|Trilinears|Tripolars"
    r"|Lines|Note|Also|Polar|Coordinates|equals?|Compare|The\s)",
    re.IGNORECASE,
)

def _extract_bary_line(lines):
    """
    Return the coordinate string from the first 'Barycentrics ...' line,
    stripped of prose trailing text.  Returns None if not found.
    """
    bary_pat = re.compile(r"^\s*Barycentrics\s*(?:=\s*)?(.+)", re.IGNORECASE)
    for line in lines:
        m = bary_pat.match(line)
        if not m:
            continue
        raw = m.group(1).strip().rstrip(",.;")
        if ":" not in raw:
            continue
        raw = _BARY_STOP.split(raw, maxsplit=1)[0].strip().rstrip(",.;")
        if ":" not in raw:
            continue
        return raw
    return None


def expand_double_colon(raw):
    """
    Expand the ETC shorthand  'u : :'  to  'u : v : w'
    via cyclic substitution  A->B->C->A,  a->b->c->a,  SA->SB->SC->SA.

    Convention:  the :: shorthand means the second and third weights are
    obtained by applying the cyclic permutation once and twice respectively.
    If raw already contains three non-empty parts, it is returned unchanged.
    """
    parts = [p.strip() for p in raw.split(":")]
    # Already a complete triple
    if len(parts) >= 3 and all(p for p in parts[:3]):
        return raw
    # Shorthand: first part non-empty, second and third empty (or missing)
    u = parts[0].strip() if parts else ""
    rest_empty = all(not p.strip() for p in parts[1:])
    if not u or not rest_empty:
        return raw

    # Sentinel-based cyclic substitution (avoids cascade).
    # Longer tokens first; trig function names and 'pi' are invariant.
    SUBS = [
        ("SA", "\x01", "SB", "SC"),
        ("SB", "\x02", "SC", "SA"),
        ("SC", "\x03", "SA", "SB"),
        ("sin","\x10","sin","sin"), ("cos","\x11","cos","cos"),
        ("tan","\x12","tan","tan"), ("cot","\x13","cot","cot"),
        ("sec","\x14","sec","sec"), ("csc","\x15","csc","csc"),
        ("pi", "\x16","pi", "pi"),
        ("A",  "\x04","B",  "C"),
        ("B",  "\x05","C",  "A"),
        ("C",  "\x06","A",  "B"),
        ("a",  "\x07","b",  "c"),
        ("b",  "\x08","c",  "a"),
        ("c",  "\x09","a",  "b"),
    ]

    def cycle(s, step):
        for src, sentinel, c1, c2 in SUBS:
            if len(src) == 1:
                s = re.sub(
                    rf"(?<![A-Za-z0-9_]){re.escape(src)}(?![A-Za-z0-9_])",
                    sentinel, s,
                )
            else:
                s = s.replace(src, sentinel)
        for src, sentinel, c1, c2 in SUBS:
            s = s.replace(sentinel, c1 if step == 1 else c2)
        return s

    return f"{u} : {cycle(u, 1)} : {cycle(u, 2)}"


def _clean_func_body(body):
    """
    Pre-clean a functional body string extracted from ETC get_text output.

    get_text() renders  <sup>2</sup>  as a space-separated digit 'a 2',
    and ETC uses square brackets as grouping brackets [expr].
    These must be fixed before string_expand_func uses the body as a template.

    Implicit multiplication (e.g. 'a(b+c)') is left for clean_bary to handle.
    """
    body = body.replace("[", "(").replace("]", ")")        # [ ] -> ( )
    body = re.sub(                                          # 'a 2' -> 'a**2'
        r"(?<![A-Za-z0-9_])([A-Za-z])\s+(\d+)(?![A-Za-z(])", r"\1**\2", body)
    body = re.sub(r"\)\s+(\d+)(?![A-Za-z(])", r")**\1", body)  # ') 2' -> ')**2'
    body = body.replace("^", "**")                         # ^ -> **
    return body.strip()


def _extract_funcs(lines):
    """
    Find lines of the form  f(a,b,c) = <body>  and return a dict
    fname -> (v1, v2, v3, body_str).

    The body is pre-cleaned by _clean_func_body so that superscript digits,
    square brackets, and carets are normalised before downstream expansion.
    """
    funcs = {}
    block = " ".join(lines)
    for m in re.finditer(
        r"\b([fghFGH])\s*\(\s*([a-zA-Z])\s*,\s*([a-zA-Z])\s*,\s*([a-zA-Z])\s*\)"
        r"\s*=\s*(.*?)(?=\s*(?:;|\bfor\b|\bwhere\b|\bBarycentrics\b|\bTrilinears\b)|$)",
        block, re.IGNORECASE,
    ):
        fname, v1, v2, v3, body = m.groups()
        body = _clean_func_body(body.strip().rstrip(",.;"))
        if body:
            funcs[fname.lower()] = (v1, v2, v3, body)
    return funcs


# Patterns that mark cross-reference or prose text after "Barycentrics"
# e.g. "Barycentrics for X(1113): ..." or "Barycentrics of the incenter..."
# The content after "Barycentrics" must be a DIRECT coordinate expression,
# not a note or cross-reference.
_BARY_PROSE_START = re.compile(
    r"^(?:for|of|are|see|that|in|at|to|from|"
    r"with|and|or|is|the|an?|by)",
    re.IGNORECASE,
)
_BARY_CROSS_REF = re.compile(r"X\s*\(\s*\d", re.IGNORECASE)


# Known mathematical function names — used by the prose-detection validator
_MATH_WORDS = frozenset([
    'sin','cos','tan','cot','sec','csc','sinh','cosh','tanh',
    'sqrt','cbrt','Sqrt','acot','atan','asin','acos','atan2',
    'exp','log','abs','Abs','sgn','sign','floor','ceil',
    # Conway S-symbols (all uppercase, 2 chars)
    # are handled by the single-char / operator checks
])


def _is_math_coord(raw):
    """
    Return True if `raw` looks like a mathematical coordinate triple.

    A genuine barycentric coordinate (e.g. "b + c - a : c + a - b : a + b - c",
    "cot A/2 : cot B/2 : cot C/2", "S^2 - SB SC : :") contains:
    - Mathematical operators (+, -, *, /, ^, parentheses) between tokens, OR
    - Known trig/function names as multi-character alphabetic tokens.

    English prose (e.g. "semi-major axis of A-Steiner ellipse") contains
    multiple plain English words ≥ 4 chars that are NOT function names.
    We reject a candidate if any colon-separated component contains 2+ such words.
    """
    for part in raw.split(':'):
        words = re.findall(r'[A-Za-z]{4,}', part)   # words of length ≥ 4
        prose = [w for w in words if w.lower() not in _MATH_WORDS]
        if len(prose) >= 2:
            return False
    return True


def _extract_all_bary_lines(lines):
    """
    Extract ALL genuine Barycentrics coordinate strings from `lines`.

    `lines` comes from _node_to_lines which already stops at <p>/<hr>,
    so most prose is already excluded.  These filters are belt-and-suspenders.

    Rejection rules (in order):
      1. Line must start with "Barycentrics" followed by whitespace.
      2. Content must NOT begin with a prose connective word
         (for, of, are, see, that, in, at, the, and, by, from, to, is).
         NOTE: "an?" intentionally absent — "a" is side-length barycentric.
      3. Content must NOT contain a center cross-reference X(N).
      4. After stop-word trimming, at least one colon must remain.
      5. Math validator: no colon-segment may contain 2+ plain English words
         of length ≥ 4 that are not known math/trig function names.
         This blocks prose like "semi-major axis of A-Steiner ellipse : :".
    """
    _bary_re = re.compile(r"^Barycentrics\s+(.+)", re.IGNORECASE)
    _prose   = re.compile(
        r"^(?:for|of|are|see|that|in|at|the|and|by|from|to|is)",
        re.IGNORECASE,
    )
    _xref    = re.compile(r"X\s*\(\s*\d", re.IGNORECASE)

    results = []
    for line in lines:
        m = _bary_re.match(line)
        if not m:
            continue
        raw = m.group(1).strip().rstrip(",.;")
        if ":" not in raw:
            continue
        if _prose.match(raw):
            continue
        if _xref.search(raw):
            continue
        raw = _BARY_STOP.split(raw, maxsplit=1)[0].strip().rstrip(",.;")
        if ":" not in raw:
            continue
        if not _is_math_coord(raw):          # reject prose like "semi-major axis..."
            continue
        results.append(expand_double_colon(raw))
    return results

def extract_bary_and_funcs(center_node):
    """
    Extract ALL Barycentrics coordinate strings and functional definitions
    for this triangle center.

    Returns (bary_list, funcs_dict):
      bary_list -- list of raw coordinate strings (each with :: expanded).
                   Empty list if no Barycentrics line is found.
      funcs     -- dict  fname -> (v1, v2, v3, body_str)

    Having all representations available lets the AST Complexity Scorer in
    Section 11 route each center through the algebraically cheapest path:
      - Pure polynomial (a, b, c, SA, SB)  → preferred (no trig penalty)
      - Trigonometric (sin, cos, tan)       → ×10 cost multiplier

    Funcs filtering:
    Only funcs whose name appears in at least one bary string are kept, to
    avoid contamination from cross-reference notes in the HTML block.
    """
    lines     = _node_to_lines(center_node)
    bary_list = _extract_all_bary_lines(lines)
    funcs     = _extract_funcs(lines)

    # Filter: keep only funcs actually called in one of the bary strings
    if funcs:
        all_barys_joined = " ".join(bary_list)
        if all_barys_joined:
            funcs = {
                fname: defn for fname, defn in funcs.items()
                if re.search(rf'\b{re.escape(fname)}\s*\(',
                             all_barys_joined, re.IGNORECASE)
            }
        else:
            funcs = {}

    return bary_list, funcs


### 5 · Fetch All 36 Pages

In [5]:
all_centers = []
fetch_log = []

for i, page in enumerate(pages, 1):
    page_name = page.split('/')[-1]
    print(f"[{i:02d}/36] Fetching {page_name} ...", end=" ")
    html = get_page(page)
    if html is None:
        print("SKIP")
        fetch_log.append((page_name, False, 0))
        continue

    centers = parse_centers(html)
    for c in centers:
        bary_list, funcs = extract_bary_and_funcs(c["node"])
        all_centers.append({
            "index": c["index"],
            "name": c["name"],
            "source_page": page_name,
            "bary_list": bary_list,   # list of all representations
            "funcs": funcs,
        })
    fetch_log.append((page_name, True, len(centers)))
    print(f"{len(centers)} centers  (running total: {len(all_centers)})")

fetched_ok = sum(1 for _, ok, _ in fetch_log if ok)
print(f"\nDone -- {len(all_centers)} centers collected.")
print(f"Pages fetched successfully: {fetched_ok}/{len(pages)}")
if fetched_ok != len(pages):
    missing = [name for name, ok, _ in fetch_log if not ok]
    print("Missing pages:", ", ".join(missing))


[01/36] Fetching ETC.html ... 998 centers  (running total: 998)
[02/36] Fetching ETCPart2.html ... 2000 centers  (running total: 2998)
[03/36] Fetching ETCPart3.html ... 2000 centers  (running total: 4998)
[04/36] Fetching ETCPart4.html ... 2000 centers  (running total: 6998)
[05/36] Fetching ETCPart5.html ... 2999 centers  (running total: 9997)
[06/36] Fetching ETCPart6.html ... 2000 centers  (running total: 11997)
[07/36] Fetching ETCPart7.html ... 2000 centers  (running total: 13997)
[08/36] Fetching ETCPart8.html ... 2000 centers  (running total: 15997)
[09/36] Fetching ETCPart9.html ... 2000 centers  (running total: 17997)
[10/36] Fetching ETCPart10.html ... 2000 centers  (running total: 19997)
[11/36] Fetching ETCPart11.html ... 2000 centers  (running total: 21997)
[12/36] Fetching ETCPart12.html ... 2000 centers  (running total: 23997)
[13/36] Fetching ETCPart13.html ... 2000 centers  (running total: 25997)
[14/36] Fetching ETCPart14.html ... 2000 centers  (running total: 27997)

In [6]:
import pandas as pd

df = pd.DataFrame(all_centers)
print(f"DataFrame shape: {df.shape}")

# Count representations per center
n_reprs = df["bary_list"].apply(len)
print(f"\nBarycentrics representations per center:")
print(f"  0 (no bary)  : {(n_reprs==0).sum()}")
print(f"  1 (single)   : {(n_reprs==1).sum()}")
print(f"  2 (two)      : {(n_reprs==2).sum()}")
print(f"  3+ (multiple): {(n_reprs>=3).sum()}")

df[["index","name","bary_list","funcs"]].head(10)


DataFrame shape: (71746, 4)

Barycentrics representations per center:
  0 (no bary)  : 5720
  1 (single)   : 63704
  2 (two)      : 2043
  3+ (multiple): 279


,index,name,bary_list,funcs
0,1,INCENTER,"[a : b : c, sin A : sin B : sin C]",{}
1,2,CENTROID,[1 : 1 : 1],{}
2,3,CIRCUMCENTER,"[sin 2A : sin 2B : sin 2C, tan B + tan C : tan...",{}
3,4,ORTHOCENTER,"[1/SA : 1/SB : 1/SC, tan A : tan B : tan C, 1/...",{}
4,5,NINE-POINT CENTER,"[a cos(B - C) : b cos(C - A) : c cos(A - B), S...",{}
5,6,"SYMMEDIAN POINT (LEMOINE POINT, GREBE POINT)","[a 2 : b 2 : c 2, SB + SC : SC + SA : SA + SB,...",{}
6,7,GERGONNE POINT,[1/(b + c - a) : 1/(c + a - b) : 1/(a + b - c)...,{}
7,8,NAGEL POINT,"[b + c - a : c + a - b : a + b - c, cot A/2 : ...",{}
8,9,MITTENPUNKT,"[a(b + c - a) : b(c + a - b) : c(a + b - c), 1...",{}
9,10,SPIEKER CENTER,[b + c : c + a : a + b],{}


### 6 · String-Cleaning Helpers (`clean_bary`)

Defines `clean_bary(expr, funcs)` and its helpers as **module-level functions only**.
The function is called inside `_compute_one` (Cell 11), not here, so the pipeline
is robust to cells being re-run out of order.

Key cleaning steps applied in order:
1. Unicode → ASCII  (`π→pi`, `−→-`, `²→**2`, …)
2. Superscript digits from `get_text`: `a 2` → `a**2`
3. Expand abstract `f(a,b,c)` using the definitions extracted in Cell 4
4. Caret `^` → `**`
5. Trig normalisation: `sin**2(A)→sin(A)**2`, `sin A→sin(A)`, `sin 2A→sin(2*A)`
6. Implicit multiplication: `a(b+c)→a*(b+c)`, `2SA→2*SA`, `SASB→SA*SB`
7. Side products: `bc→b*c`, Conway underscores: `S_A→SA`

In [7]:
import re
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
)

# ── String-cleaning helpers ───────────────────────────────────────────────────
# (These are defined here as module-level functions; clean_bary is called
#  inside _compute_one so the pipeline is robust to cell execution order.)

_UNICODE_MAP = {
    '²': '**2', '³': '**3', '⁴': '**4', '⁵': '**5',
    '·': '*', '−': '-', '–': '-', ' ': ' ',
    'π': 'pi', 'α': 'alpha', 'β': 'beta', 'γ': 'gamma'
}
_TRIG        = r'(sin|cos|tan|cot|sec|csc)'
_KNOWN_FUNCS = frozenset(['sin','cos','tan','cot','sec','csc','sqrt','Abs','pi','exp','log'])
_ABSTRACT_FN = re.compile(r'\b[fghFGH]\s*\(')


def _insert_impl_mul(s):
    """Insert * before ( when preceded by a non-function identifier."""
    def rep(m):
        return m.group(1) + ('(' if m.group(1) in _KNOWN_FUNCS else '*(')
    return re.sub(r'\b([A-Za-z_][A-Za-z0-9_]*)\(', rep, s)


def string_expand_func(expr, fname, v1, v2, v3, body):
    """
    Expand all occurrences of  fname(arg1, arg2, arg3)  by substituting
    the formal parameters (v1, v2, v3) with the actual arguments.

    Sentinel characters \x01\x02\x03 prevent cascade substitution when the
    arguments themselves contain the parameter names.
    """
    if not expr: return expr
    pattern = rf'\b{re.escape(fname)}\s*\(([^,)]+),\s*([^,)]+),\s*([^)]+)\)'
    def repl(m):
        a1, a2, a3 = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        res = body
        S1, S2, S3 = '\x01', '\x02', '\x03'
        res = re.sub(rf'\b{re.escape(v1)}\b', S1, res)
        res = re.sub(rf'\b{re.escape(v2)}\b', S2, res)
        res = re.sub(rf'\b{re.escape(v3)}\b', S3, res)
        return f'({res.replace(S1, f"({a1})").replace(S2, f"({a2})").replace(S3, f"({a3})")})'
    return re.sub(pattern, repl, expr)


def clean_bary(text):
    if not isinstance(text, str): return ""
    
    # Unicode and common replacements
    for k, v in _UNICODE_MAP.items():
        text = text.replace(k, v)
    text = text.replace('^', '**')
    
    # Conway and Sw (Conway symbols are handled first to avoid 's' conflicts)
    text = text.replace('S_A', 'SA').replace('S_B', 'SB').replace('S_C', 'SC')
    text = re.sub(r'S_\\omega|S_w|Somega', 'Sw', text)
    
    # Trig regex: sin 2A -> sin(2*A), sin^2 A -> sin(A)**2
    # These use \b (word boundaries) to protect variables like 's' or 'r'
    text = re.sub(r'(sin|cos|tan|cot|sec|csc)\*\*2\s*([A-C])', r'\1(\2)**2', text)
    text = re.sub(r'\b(sin|cos|tan|cot|sec|csc)\s+([A-C])\b', r'\1(\2)', text)
    text = re.sub(r'\b(sin|cos|tan|cot|sec|csc)\s+([A-C])/(\d+)\b', r'\1(\2/\3)', text)
    
    return text.strip()


### 7 · Geometry Definitions for the Right-Triangle Setup

In [8]:
import sympy as sp
import math

# ── Symbols ───────────────────────────────────────────────────────────────────
theta, t = sp.symbols('theta t', real=True)
C_ang     = sp.Symbol('C', real=True, positive=True)  # kept symbolic for limits
w_sym     = sp.Symbol('w', positive=True)              # Brocard angle

S, SA, SB, SC = sp.symbols('S SA SB SC', real=True)
Sw, s          = sp.symbols('Sw s',       real=True)
sa, sb, sc     = sp.symbols('sa sb sc',   real=True)
r, R           = sp.symbols('r R',        positive=True)

# ── Vertex angles ─────────────────────────────────────────────────────────────
#   A=(1,0), B=(-1,0), C=(cos theta, sin theta) on the unit circumcircle
#   By the inscribed-angle theorem:
#     angle A = (pi - theta)/2,  angle B = theta/2,  angle C = pi/2
#
#   C is placed directly in geom_sub as pi/2.
#   Expressions like  sin(C)=1, cos(C)=0, tan(C)=zoo  are evaluated immediately.
#   A zoo (complex infinity) result means the center has a pole at C=pi/2
#   for our right-triangle setup -- caught in _compute_one as "degen:C_pole".
#   This is faster and more robust than calling sp.limit for every center.
A_angle = (sp.pi - theta) / 2
B_angle = theta / 2
C_angle = sp.pi / 2   # placed in geom_sub

# ── Side lengths ──────────────────────────────────────────────────────────────
a_side = 2 * sp.cos(theta / 2)   # BC (opposite A)
b_side = 2 * sp.sin(theta / 2)   # CA (opposite B)
c_side = sp.Integer(2)           # AB = 2R = 2  (hypotenuse / diameter)

# ── Conway symbols ────────────────────────────────────────────────────────────
a2, b2, c2 = a_side**2, b_side**2, c_side**2
SA_val = (b2 + c2 - a2) / 2     # = 2(1 - cos theta)
SB_val = (c2 + a2 - b2) / 2     # = 2(1 + cos theta)
SC_val = sp.Integer(0)           # right-angle at C -> a^2+b^2 = c^2

# S = 2*Area.
#   Area = (1/2)*a*b*sin(C) = (1/2)*a*b   so  S = a*b = 2*sin(theta)
S_val  = a_side * b_side

s_val  = (a_side + b_side + c_side) / 2
r_val  = S_val / (2 * s_val)     # inradius = Area/s = S/(2s)
R_val  = sp.Integer(1)           # circumradius = 1
Sw_val = SA_val + SB_val         # S_omega = SA+SB+SC = SA+SB  (SC=0)

# Brocard angle w:
#   General formula:  cot(w) = (a^2+b^2+c^2) / (2S)
#   For our right triangle: (a^2+b^2+c^2)/(2S) = 8/(2*2sin(theta)) = 2/sin(theta)
#   Also: cot(w) = cot(A)+cot(B)+cot(C)  with cot(pi/2)=0
#   => cot(w) = tan(theta/2) + cot(theta/2)  [via double-angle identities]
cotw_val = (a2 + b2 + c2) / (2 * S_val)   # = 2/sin(theta)
w_val    = sp.acot(cotw_val)               # atan(sin(theta)/2)

# ── Substitution dictionary ───────────────────────────────────────────────────
# C_ang -> pi/2 is included directly.
#   - sin(C)=1, cos(C)=0 substitute cleanly.
#   - tan(C)=zoo, sec(C)=zoo: these produce zoo in the output, which is
#     caught by the "degen:C_pole" check in _compute_one.
#   - This replaces the former sp.limit(expr, C_ang, pi/2) slow path,
#     which was the primary cause of 3-5 hour runtimes on complex inputs.
geom_sub = {
    sp.Symbol('a'): a_side,
    sp.Symbol('b'): b_side,
    sp.Symbol('c'): c_side,
    sp.Symbol('A'): A_angle,
    sp.Symbol('B'): B_angle,
    C_ang:          sp.pi / 2,   # direct substitution; zoo caught downstream
    S: S_val, SA: SA_val, SB: SB_val, SC: SC_val, Sw: Sw_val,
    s: s_val,
    sa: s_val - a_side, sb: s_val - b_side, sc: s_val - c_side,
    r: r_val, R: R_val,
    # Brocard angle: cot and tan substituted directly; sin/cos handled via w_val
    sp.cot(w_sym): cotw_val,
    sp.tan(w_sym): sp.Integer(1) / cotw_val,
    w_sym: w_val,
}

# ── Rational parametrisation  t = tan(theta/4) ───────────────────────────────
THETA_T = 4 * sp.atan(t)

_s2  = 2*t / (1 + t**2)
_c2  = (1 - t**2) / (1 + t**2)
_sth = sp.cancel(2 * _s2 * _c2)
_cth = sp.cancel(_c2**2 - _s2**2)

rat_sub_dict = {
    sp.sin(theta/2):  _s2,
    sp.cos(theta/2):  _c2,
    sp.tan(theta/2):  sp.cancel(_s2 / _c2),
    sp.cot(theta/2):  sp.cancel(_c2 / _s2),
    sp.sin(theta):    _sth,
    sp.cos(theta):    _cth,
    sp.tan(theta):    sp.cancel(_sth / _cth),
    sp.cot(theta):    sp.cancel(_cth / _sth),
}

print("Geometry setup complete.")
print(f"  a  = {a_side}")
print(f"  b  = {b_side}")
print(f"  S  = {sp.simplify(S_val)}   (Conway S = 2*Area)")
print(f"  SA = {sp.simplify(SA_val)}")
print(f"  SB = {sp.simplify(SB_val)}")
print(f"  SC = {SC_val}              (right angle at C)")
print(f"  cot(w) = {sp.simplify(cotw_val)}")
print(f"  Note: C=pi/2 in geom_sub; tan(C)/sec(C) poles caught as degen:C_pole")

import numpy as np

import math
from sympy.functions.elementary.trigonometric import TrigonometricFunction


def detect_denominators(expr):
    """
    Return list of all denominators d found in theta/d trig arguments.

    Used to determine the optimal Weierstrass substitution parameter.
    The LCD of the returned list = n  gives  t = tan(theta/(2n)).

    Design: uses as_coefficient(theta) which is O(n) and exact, unlike
    sp.simplify(arg/theta) which is slow and can fail on multi-term args.
    """
    denoms = []
    for node in expr.find(TrigonometricFunction):
        arg = node.args[0]
        for term in sp.Add.make_args(sp.expand(arg)):
            c = term.as_coefficient(theta)
            if c is not None and c.is_Rational and c != 0:
                denoms.append(int(abs(c).q))
    return denoms if denoms else [1]


def choose_weierstrass_n(denoms):
    """
    Choose the parameter n for  t = tan(theta / (2n)).

    Pure powers of 2: n = max(denoms) = 2^k
      Example: denoms=[1,2,4] -> n=4, t=tan(theta/8)
    Mixed prime factors: n = lcm(all denoms)
      Example: denoms=[2,3] -> n=6, t=tan(theta/12)

    Both cases are unified by n = lcm(denoms), since:
      lcm(2^a, 2^b) = 2^max(a,b)  (the power-of-2 case)
      lcm(p,q) = p*q/gcd(p,q)     (the mixed case)

    With t = tan(theta/(2n)), every sin(theta/d) for d | n is rational in t
    because theta/d = (n/d)*(theta/n) and sin(k*base) is a polynomial in
    sin(base) and cos(base), each rational in tan(base/2) = t.
    """
    return math.lcm(*denoms)


def build_weierstrass_sub(n):
    """
    Build a fast-path substitution dict for  t = tan(theta/(2n)).

    Covers sin/cos/tan/cot of theta/n (base) and theta/(n/2) (double base)
    directly; compound multiples are handled by the general rewrite(tan) path.

    Returns (THETA_T, rat_sub_dict) where THETA_T = 2n*atan(t).
    """
    THETA_T_n = sp.Integer(2) * n * sp.atan(t)
    u   = theta / n                        # base angle
    su  = 2*t      / (1 + t**2)            # sin(u) = sin(theta/n)
    cu  = (1-t**2) / (1 + t**2)            # cos(u)
    s2u = sp.cancel(2 * su * cu)           # sin(2u)
    c2u = sp.cancel(cu**2 - su**2)         # cos(2u)
    rsd = {
        sp.sin(u): su,   sp.cos(u): cu,
        sp.tan(u): sp.cancel(su/cu),   sp.cot(u): sp.cancel(cu/su),
        sp.sin(2*u): s2u, sp.cos(2*u): c2u,
        sp.tan(2*u): sp.cancel(s2u/c2u), sp.cot(2*u): sp.cancel(c2u/s2u),
    }
    return THETA_T_n, rsd




def is_degenerate_denom(u_th, v_th, w_th):
    """
    Return True if u + v + w is identically zero after geometry substitution.

    A zero sum means the three weights define a point at projective infinity --
    no finite Cartesian coordinates exist.  Detecting this early prevents a
    ZeroDivisionError (or zoo result) deep inside the pipeline.
    """
    return sp.cancel(u_th + v_th + w_th) == 0


def make_numpy_func(expr_t):
    """
    Wrap sp.lambdify so that the returned callable always produces a
    1-D numpy array, even when expr_t is a numeric constant.

    sp.lambdify of a constant (e.g. X(3) Circumcenter x = 0) returns a
    Python int/float, not an array.  Broadcasting is needed before
    downstream code can treat all x_func / y_func uniformly.
    """
    raw = sp.lambdify(t, expr_t, modules='numpy')

    def f(t_vals):
        t_arr = np.asarray(t_vals, dtype=float)
        result = raw(t_arr)
        if np.ndim(result) == 0:              # constant expression
            return np.full(t_arr.shape, float(result))
        return np.asarray(result, dtype=float)

    return f


Geometry setup complete.
  a  = 2*cos(theta/2)
  b  = 2*sin(theta/2)
  S  = 2*sin(theta)   (Conway S = 2*Area)
  SA = 2 - 2*cos(theta)
  SB = 2*cos(theta) + 2
  SC = 0              (right angle at C)
  cot(w) = 2/sin(theta)
  Note: C=pi/2 in geom_sub; tan(C)/sec(C) poles caught as degen:C_pole


### 8 · `parse_bary` — SymPy-parse the Barycentric String

In [9]:
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
)

# Enable implicit multiplication: a(b+c) -> a*(b+c), 2SA -> 2*SA
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor)

# Canonical symbols reused across all parse calls for stability + speed
_PARSE_LOCALS = {
    'a': sp.Symbol('a', real=True, positive=True),
    'b': sp.Symbol('b', real=True, positive=True),
    'c': sp.Symbol('c', real=True, positive=True),
    'A': sp.Symbol('A', real=True),
    'B': sp.Symbol('B', real=True),
    'C': C_ang,
    'SA': SA, 'SB': SB, 'SC': SC,
    'S': S, 'Sw': Sw, 'SW': Sw,
    's': s, 'sa': sa, 'sb': sb, 'sc': sc,
    'r': r, 'R': R, 'w': w_sym,
    'sin': sp.sin, 'cos': sp.cos, 'tan': sp.tan,
    'cot': sp.cot, 'sec': sp.sec, 'csc': sp.csc,
    'sqrt': sp.sqrt, 'Abs': sp.Abs, 'exp': sp.exp, 'log': sp.log,
    'pi': sp.pi,
}


def _expand_custom_functions(expr_text, custom_funcs):
    out = expr_text
    if not custom_funcs:
        return out
    for fname, defn in custom_funcs.items():
        if not (isinstance(defn, (tuple, list)) and len(defn) == 4):
            continue
        v1, v2, v3, body = defn
        out = string_expand_func(out, fname, v1, v2, v3, body)
    return out


def parse_bary(bary_str, custom_funcs=None):
    """Parse a barycentric triple string into a SymPy tuple (u, v, w)."""
    if not isinstance(bary_str, str) or ':' not in bary_str:
        return None

    expanded = _expand_custom_functions(bary_str, custom_funcs)
    parts = [p.strip() for p in expanded.split(':')]
    if len(parts) < 3:
        return None
    parts = parts[:3]
    if any(not p for p in parts):
        return None

    try:
        parsed = tuple(
            parse_expr(piece, local_dict=_PARSE_LOCALS, transformations=_TRANSF, evaluate=False)
            for piece in parts
        )
    except Exception:
        return None

    return parsed if len(parsed) == 3 else None


### 9 · `rationalize_expr` — Adaptive Weierstrass Substitution

Converts a theta-expression to a rational (or algebraic) function of `t`.

**Algorithm:**
1. `expand_trig(expr)` to expose all `theta/d` denominators
2. Collect denominators → `n = lcm(all d)` → `t = tan(theta/(2n))`
   - Pure powers of 2: `n = max(d) = 2^k`, e.g. denoms=[1,2,4] → n=4, `t=tan(θ/8)`
   - Mixed: `n = lcm(d)`, e.g. denoms=[2,3] → n=6, `t=tan(θ/12)`
3. Fast path (n≤2): direct dict substitution
4. General path: `theta → 2n·atan(t)`, then `expand_trig`, `rewrite(tan)`, substitute `tan(atan(t))→t`
5. Fallback: `trigsimp` + same rewrite chain

The n is computed **globally** across all three (u,v,w) components so that `x(t)` and `y(t)` share the same parameter.


In [10]:
# ── Fast-path substitution dict for n=2  (t = tan(θ/4)) ─────────────────────
_t_sq   = t**2
_denom  = 1 + _t_sq
_denom2 = _denom**2

_FAST_SUB_N2 = {
    sp.sin(theta/2):  2*t / _denom,
    sp.cos(theta/2):  (1 - _t_sq) / _denom,
    sp.tan(theta/2):  2*t / (1 - _t_sq),
    sp.cot(theta/2):  (1 - _t_sq) / (2*t),
    sp.sin(theta):    4*t*(1 - _t_sq) / _denom2,
    sp.cos(theta):    ((1 - _t_sq)**2 - 4*_t_sq) / _denom2,
    sp.tan(theta):    4*t*(1 - _t_sq) / ((1 - _t_sq)**2 - 4*_t_sq),
    sp.cot(theta):    ((1 - _t_sq)**2 - 4*_t_sq) / (4*t*(1 - _t_sq)),
}

# ── Pre-computed Cx(t), Cy(t) ────────────────────────────────────────────────
_CX_T = {}; _CY_T = {}

def _get_CxCy(n):
    if n not in _CX_T:
        if n == 2:
            _CX_T[2] = sp.cancel(sp.cos(theta).subs(_FAST_SUB_N2))
            _CY_T[2] = sp.cancel(sp.sin(theta).subs(_FAST_SUB_N2))
        else:
            THETA_T = sp.Integer(2)*n*sp.atan(t)
            cx = sp.expand_trig(sp.cos(theta).subs(theta, THETA_T))
            cy = sp.expand_trig(sp.sin(theta).subs(theta, THETA_T))
            _CX_T[n] = sp.cancel(cx.rewrite(sp.tan).subs(sp.tan(sp.atan(t)), t))
            _CY_T[n] = sp.cancel(cy.rewrite(sp.tan).subs(sp.tan(sp.atan(t)), t))
    return _CX_T[n], _CY_T[n]

_get_CxCy(2)   # warm up at import time


# ── Plain dict caches (picklable by loky/cloudpickle, unlike @lru_cache) ────
# @lru_cache decorated functions cannot be pickled in interactive Jupyter contexts.
# Plain dict caches are always picklable and persist within each worker process
# for its 500-task lifetime (max_tasks_per_child=500).
# maxsize enforced by pruning the oldest half when the limit is reached.

_GEOM_CACHE = {}
_RAT_CACHE  = {}
_CACHE_MAX  = 8192

def _cached_geom_expand(expr):
    """Dict-cached sp.expand_trig(expr.subs(geom_sub)) — picklable."""
    if expr not in _GEOM_CACHE:
        if len(_GEOM_CACHE) >= _CACHE_MAX:
            for k in list(_GEOM_CACHE)[:_CACHE_MAX // 2]:
                del _GEOM_CACHE[k]
        _GEOM_CACHE[expr] = sp.expand_trig(expr.subs(geom_sub))
    return _GEOM_CACHE[expr]

def _cached_rationalize(expr, n):
    """Dict-cached _rationalize_one — picklable."""
    key = (expr, n)
    if key not in _RAT_CACHE:
        if len(_RAT_CACHE) >= _CACHE_MAX:
            for k in list(_RAT_CACHE)[:_CACHE_MAX // 2]:
                del _RAT_CACHE[k]
        _RAT_CACHE[key] = _rationalize_one(expr, n)
    return _RAT_CACHE[key]


# ── AST Complexity Scorer ────────────────────────────────────────────────────

def _count_ast_nodes(expr):
    """Count the number of nodes in a SymPy expression tree (fast traversal)."""
    if not isinstance(expr, sp.Basic):
        return 0
    return sum(1 for _ in sp.preorder_traversal(expr))

def _score_coords(coords):
    """
    Heuristic cost of a barycentric triple (u, v, w).

    Lower score = cheaper to process through geom_sub + GCD pipeline.

    Scoring rules (applied to parsed SymPy, before geom_sub):
      base  = total AST nodes (reflects expression size)
      × 10  if any component contains a TrigonometricFunction
             (trig → polynomial conversion via rewrite(tan) is expensive)
      × 1.5 if any component contains Pow (rational exponents, sqrt)

    Why score BEFORE geom_sub?
    The raw parsed symbols (SA, SB, a, b, ...) reveal structure cheaply.
    After geom_sub everything becomes sin/cos of theta anyway.
    """
    base     = sum(_count_ast_nodes(c) for c in coords)
    has_trig = any(c.has(TrigonometricFunction) for c in coords)
    has_pow  = any(c.has(sp.Pow) for c in coords)
    return base * (10 if has_trig else 1) * (1.5 if has_pow else 1)

def _choose_simplest(parsed_list):
    """
    Given a list of parsed (u, v, w) triples, return the one with the
    lowest AST complexity score.
    """
    best, best_score = None, float('inf')
    for coords in parsed_list:
        if coords is None:
            continue
        s = _score_coords(coords)
        if s < best_score:
            best_score = s
            best = coords
    return best


# ── Core rationalize helpers ─────────────────────────────────────────────────

def _rationalize_one(expr, n):
    """Rationalize without cancel (caller decides when to cancel)."""
    if expr is None or not isinstance(expr, sp.Basic):
        return None
    if n == 2:
        r = expr.subs(_FAST_SUB_N2)
        if not r.has(TrigonometricFunction):
            return r
    THETA_T_n = sp.Integer(2)*n*sp.atan(t)
    sub = sp.expand_trig(expr.subs(theta, THETA_T_n))
    a2r = sub.rewrite(sp.tan).subs(sp.tan(sp.atan(t)), t)
    return a2r if not a2r.has(theta) else None

def rationalize_expr(expr_theta, n_override=None):
    """Public API: rationalize and cancel."""
    n = n_override if n_override is not None else 2
    r = _rationalize_one(expr_theta, n)
    return sp.cancel(r) if r is not None else None


### 10 · Denominator-Clearing Helper

**Purpose**: Expressions like `1/SA : 1/SB : 1/SC` have `SC = 0` in the denominator.
Multiplying all weights by `lcm(denom(u), denom(v), denom(w))` *before* geometry
substitution converts this to `SB·SC : SA·SC : SA·SB`, which correctly yields `0:0:SA·SB`
after SC → 0.

**Warning fix**: uses plain `x * y` multiplication instead of `sp.Mul(x, y, evaluate=True)`.
The latter bypasses Python's `__mul__` protocol and can pass a SymPy `Poly` Tuple directly
to `Mul.__new__`, which `radsimp` then flags as `SymPyDeprecationWarning`.


In [11]:
def _safe_poly_lcm(a, b, syms):
    """
    Compute LCM of two expressions via Poly objects.
    Poly.lcm().as_expr() always returns a clean SymPy Expr, never a Tuple.
    """
    try:
        return sp.lcm(sp.Poly(a, *syms), sp.Poly(b, *syms)).as_expr()
    except Exception:
        # Integer-GCD fallback -- also stays in Expr-land
        g = sp.gcd(a, b)
        return a * (b // g) if g != 0 else a * b


def clear_denominators(expr):
    """Lighter-weight denominator clearing using cancel() only."""
    if expr is None: return None
    try:
        # cancel() is significantly faster than simplify() for barycentrics
        n, d = sp.fraction(sp.cancel(expr))
        return n if d != 0 else expr
    except:
        return expr

def resolve_C(x_expr, y_expr):
    """
    [LEGACY - now a no-op]

    C_ang -> pi/2 is handled directly in geom_sub (Cell 7).
    After geom_sub, C_ang never appears in the Cartesian expressions.
    Centers where tan(C) or sec(C) produce zoo are caught earlier
    in _compute_one by the "degen:C_pole" check.

    sp.limit was removed as the slow path here was the primary cause
    of worker threads blocking for 3-5 hours on complex inputs.
    """
    return x_expr, y_expr


## 11 · Barycentric → Cartesian Parametric Conversion

**Architecture (v11 — full optimisation stack)**

| Layer | Technique | Benefit |
|---|---|---|
| Data | Extract **all** bary representations per center | Routes around expensive trig |
| Routing | **AST Complexity Scorer** | Picks polynomial over trig → skips heavy GCD |
| Algebra | **Individual component cancel** | Reduces GCD degree before combining |
| Caching | **Dict caches** on geom_expand + rationalize | O(1) for shared sub-expressions |
| Memory | **Batched Parallel (500/batch)** | Fresh loky workers each batch → flat memory |
| Safety | **Per-task threading timeout (20s)** | Kills runaway GCD; returns `rat:timeout` |
| Export | **Checkpointing** | Saves progress after every batch; survives kernel crash |

**Why tasks slow down at X(7000+):**
Later ETC centers use higher-degree algebraic forms. SymPy leaves short-lived AST nodes that Python's GC cannot fully reclaim. After ~1000 tasks, a worker holds ~100k extra objects. Running `Parallel` in batches of 500 restarts the loky workers after each batch, clearing accumulated SymPy memory and keeping execution speed flat.

(`max_tasks_per_child` is a `multiprocessing.Pool` parameter, not available in joblib's `Parallel`. The batched loop achieves the same effect.)


In [12]:
import time
import warnings
import pandas as pd
import sympy as sp
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# ── 1) Complexity routing + helpers ───────────────────────────────────────────
_TRIG_WORDS = ('sin', 'cos', 'tan', 'cot', 'sec', 'csc')


def bary_complexity_score(raw):
    """Lower score means algebraically cheaper candidate."""
    if not isinstance(raw, str):
        return 10**9
    s = raw.replace(' ', '')
    trig_penalty = 40 * sum(s.count(w) for w in _TRIG_WORDS)
    len_penalty = len(s)
    op_penalty = s.count('/') * 5 + s.count('**') * 3
    return trig_penalty + len_penalty + op_penalty


def _geom_eval_triplet(uvw):
    return tuple(sp.cancel(sp.expand_trig(sp.simplify(comp.subs(geom_sub)))) for comp in uvw)


def _contains_pole(uvw):
    return any(comp.has(sp.zoo, sp.oo, -sp.oo, sp.nan) for comp in uvw)


def bary_to_cartesian(uvw_t, n_val):
    """Convert barycentric weights (u,v,w) to Cartesian x(t), y(t)."""
    u, v, w = uvw_t
    denom = sp.cancel(u + v + w)
    if denom == 0:
        return None, None

    Cx, Cy = _get_CxCy(n_val)
    x_expr = sp.cancel((u - v + w*Cx) / denom)
    y_expr = sp.cancel((w*Cy) / denom)
    return x_expr, y_expr


def _compute_one(bary_list, funcs_dict):
    """Evaluate every barycentric representation and keep first successful route."""
    if not isinstance(bary_list, list) or not bary_list:
        return None, None, None, 0, "no_bary"

    checked = 0
    ranked = sorted(bary_list, key=bary_complexity_score)

    for raw_str in ranked:
        checked += 1
        cleaned = clean_bary(raw_str)
        parsed = parse_bary(cleaned, funcs_dict)
        if parsed is None:
            continue

        parsed = tuple(clear_denominators(p) for p in parsed)

        try:
            theta_triplet = _geom_eval_triplet(parsed)
        except Exception:
            continue

        if _contains_pole(theta_triplet):
            continue

        denoms = []
        for comp in theta_triplet:
            denoms.extend(detect_denominators(comp))
        auto_n = choose_weierstrass_n(denoms) if denoms else 2

        n_candidates = []
        for n_val in [auto_n, 2, 4, 1]:
            if n_val not in n_candidates:
                n_candidates.append(n_val)

        for n_val in n_candidates:
            try:
                rat_triplet = tuple(rationalize_expr(comp, n_override=n_val) for comp in theta_triplet)
                if any(val is None for val in rat_triplet):
                    continue
                x_t, y_t = bary_to_cartesian(rat_triplet, n_val)
                if x_t is None or y_t is None:
                    continue
                if theta in x_t.free_symbols or theta in y_t.free_symbols:
                    continue
                return x_t, y_t, n_val, checked, "ok"
            except Exception:
                continue

    return None, None, None, checked, "no_valid_route"


def _make_task_key(bl, fn):
    bl_key = tuple(bl) if isinstance(bl, list) else tuple()
    if isinstance(fn, dict):
        fn_key = tuple(sorted((k, tuple(v)) for k, v in fn.items()))
    else:
        fn_key = tuple()
    return bl_key, fn_key


def _run_task(task):
    key, bl, fn = task
    return key, _compute_one(bl, fn)


# ── 2) Prepare + deduplicate tasks ────────────────────────────────────────────
print("Preparing and deduplicating tasks...")
unique_tasks = {}
for _, row in df.iterrows():
    key = _make_task_key(row.get("bary_list", []), row.get("funcs", {}))
    if key not in unique_tasks:
        unique_tasks[key] = (row.get("bary_list", []), row.get("funcs", {}))

task_list = [(k, bl, fn) for k, (bl, fn) in unique_tasks.items()]
print(f"Unique bary/function bundles: {len(task_list)}")

# ── 3) Execute in parallel ────────────────────────────────────────────────────
t0 = time.time()
results = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(_run_task)(task) for task in task_list
)

# ── 4) Map back to dataframe ─────────────────────────────────────────────────
cache = {key: res for key, res in results}
rows_out = []
for _, row in df.iterrows():
    key = _make_task_key(row.get("bary_list", []), row.get("funcs", {}))
    rows_out.append(cache.get(key, (None, None, None, 0, "no_task")))

out_df = pd.DataFrame(rows_out, columns=["x(t)", "y(t)", "weierstrass_n", "bary_checked", "eval_status"])
for col in out_df.columns:
    df[col] = out_df[col].values

elapsed = (time.time() - t0) / 60
success_count = df["x(t)"].notna().sum()

print("\nPipeline complete.")
print(f"Successfully calculated: {success_count} / {len(df)}")
print(f"Average barycentrics checked per center: {df['bary_checked'].fillna(0).mean():.2f}")
print(f"Total time elapsed: {elapsed:.2f} minutes")


Normalizing and deduplicating tasks...
Total entries in DF: 71746
Unique mathematical tasks: 64566
Preparing and deduplicating tasks...
Processing 64566 unique centers in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


NameError: name '_run_task' is not defined

### 12 · Results

In [ ]:
display(df[["index","name","weierstrass_n","x(t)","y(t)"]].dropna(subset=["x(t)"]).head(15))
print(f"\nWeierstrass n distribution:")
print(df["weierstrass_n"].value_counts().sort_index().to_string())

In [ ]:
# --- Section 12: Conversion failure diagnostics ---
from collections import Counter

print(f"--- Pipeline Diagnostics ---")
if 'parse_errors' in locals() and parse_errors:
    reasons = [e[2] for e in parse_errors]
    counter = Counter(reasons)
    print(f"Top failure reasons ({len(parse_errors)} total):")
    for reason, count in counter.most_common(10):
        print(f"  {count:5d}  {reason}")
else:
    print("No parse errors found or mapping not run.")

print(f"\nWeierstrass n usage (t = tan(θ/(2n))):")
if "weierstrass_n" in df.columns:
    n_counts = df["weierstrass_n"].value_counts().sort_index()
    for n_val, cnt in n_counts.items():
        if isinstance(n_val, (int, float)):
            bar = "█" * min(40, int(cnt / max(n_counts.values) * 40))
            print(f"  n={n_val}  {cnt:6d} centers  {bar}")

# Preview results
display(df[["index", "name", "weierstrass_n", "x(t)", "y(t)"]].dropna(subset=["x(t)"]).head(10))

### 13 · Export to CSV

In [ ]:
# --- Section 13: Robust CSV Export ---
output_csv = "ETC_Centers_Final_Table.csv"

# 1. Create a copy for export
export_df = df.copy()

# 2. Convert SymPy expressions to clean strings
# Without this, the CSV will be filled with unreadable object memory addresses
export_df["x(t)"] = export_df["x(t)"].apply(lambda val: str(val) if val is not None else "")
export_df["y(t)"] = export_df["y(t)"].apply(lambda val: str(val) if val is not None else "")

# 3. Save
export_df.to_csv(output_csv, index=False)
print(f"Successfully exported {len(export_df)} rows to: {output_csv}")

---
## Appendix: Notation Reference

### Triangle setup
| Symbol | Value |
|--------|-------|
| A | (1, 0) |
| B | (−1, 0) |
| C | (cos θ, sin θ) |
| a (= BC) | 2 cos(θ/2) |
| b (= CA) | 2 sin(θ/2) |
| c (= AB) | 2 |
| angle A | (π − θ)/2 |
| angle B | θ/2 |
| angle C | π/2 |
| R | 1 |

### Conway symbols
| Symbol | Value |
|--------|-------|
| SA | (b²+c²−a²)/2 = 2(1−cos θ) |
| SB | (c²+a²−b²)/2 = 2(1+cos θ) |
| SC | (a²+b²−c²)/2 = **0** (right angle) |
| S  | area = sin θ |
| Sω | SA+SB+SC = SA+SB |

### Rational parametrisation  t = tan(θ/4)
| Expression | In t |
|-----------|------|
| sin(θ/2)  | 2t/(1+t²) |
| cos(θ/2)  | (1−t²)/(1+t²) |
| sin θ     | 4t(1−t²)/(1+t²)² |
| cos θ     | (1−6t²+t⁴)/(1+t²)² |

### ETC notation quirks handled by `clean_bary`
| Raw ETC string | Meaning | After cleaning |
|----------------|---------|----------------|
| `sin 2A` | sin(2A) | `sin(2*A)` |
| `a cos(B-C)` | a·cos(B−C) | `a*cos(B-C)` |
| `sin^2 A` | sin²A | `sin(A)**2` |
| `S_A` | SA (Conway) | `SA` |
| `1/SA:1/SB:1/SC` | cleared → `SB*SC:SA*SC:SA*SB` | correct limit at SC=0 |


### Weierstrass substitution parameter n

The column `weierstrass_n` records which half-angle substitution was used:

| n | substitution | typical barycentrics |
|---|---|---|
| 1 | t = tan(θ/2) | purely polynomial (1:1:1, SA:SB:SC) |
| 2 | t = tan(θ/4) | standard (a:b:c, SA:SB, sin 2A:...) |
| 4 | t = tan(θ/8) | half-angle (sin(A/2):sin(B/2):sin(C/2)) |
| 3,6,… | exotic | sin(A/3) type (rare in ETC) |

For n=2, the parametrisation `t = tan(θ/4)` is the standard Weierstrass substitution where a = 2cos(θ/2) = 2(1-t²)/(1+t²) and b = 2sin(θ/2) = 4t/(1+t²).